In [12]:
import os
import json
import pandas as pd
from IPython.display import display

In [13]:
PATH_HAMLET  = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_validated"
PATH_NORM = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_normalization_v2_20260828"
PATH_OUTPUT = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\output"
PATH_COMBINED = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\agentic-metadata\run_metadata_combined.tsv"
PATH_MLM = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\agentic-metadata\run_meta_mlmarker.tsv"
PATH_LLM = r"C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\agentic-metadata\agent_metadata.tsv"

## Biological agent  
- age
- anatomic_site_tumor (not necessary)
- cell_line
- cell_type
- developmental_stage
- disease_state
- material_type
- sample_source (can be multiple)
- sex
- species (can be multiple) 
- tissue
- _confidence (split: overall, evidence_score, completness, format_score)

## Design agent  
- technology_type
- confidence (rename to experimental_design_conf......) 

## Todo for handle_experimental_design_agent 
- Add the source shard in a new columns
- Select all column
- Do not keep the evidence in columns 

In [14]:
TARGET_COLUMNS_BIOLOGICAL = [
    "age",
    "cell_line",
    "cell_type",
    "developmental_stage",
    "disease_state",
    "material_type",
    "sample_source",
    "sex",
    "species",
    "tissue"
]
TARGET_COLUMNS = TARGET_COLUMNS_BIOLOGICAL

def extract_field_values(val, use_normalized=False):
    if not isinstance(val, list):
        return val
    extracted = []
    for item in val:
        if isinstance(item, dict):
            if use_normalized:
                val_raw = item.get("value")
                ont_name = item.get("ontology_name") if item.get("is_normalized") and item.get("ontology_name") else False
                extracted.append({"value": val_raw, "ontology_name": ont_name})
            elif "value" in item:
                extracted.append(item["value"])
            elif "ontology_name" in item:
                extracted.append(item["ontology_name"])
    if len(extracted) == 0:
        return None
    elif len(extracted) == 1:
        return extracted[0]
    return extracted

def handle_biological_agent(agent_path, filter_columns=True, use_normalized=False):
    print(f"Handling BiologicalAgent at {agent_path}")
    shard_name = os.path.basename(os.path.dirname(agent_path.rstrip(r"\/")))
    if shard_name == "NormalizedAgent":
        shard_name = os.path.basename(os.path.dirname(os.path.dirname(agent_path.rstrip(r"\/"))))
    rows = []

    for filename in sorted(os.listdir(agent_path)):
        if not filename.endswith(".json"):
            continue

        json_path = os.path.join(agent_path, filename)
        with open(json_path, "r", encoding="utf-8") as file:
            record = json.load(file)

        row = {
            "source_file": filename.split('_')[0],
            "source_shard": shard_name
        }

        # Extract values without evidence (filtered to target columns or keeping all columns)
        if filter_columns:
            for col in TARGET_COLUMNS_BIOLOGICAL:
                row[col] = extract_field_values(record.get(col), use_normalized=use_normalized)
        else:
            for key, val in record.items():
                if key in ["_confidence", "_hallucination_flags", "_evidence_grounding_flags"]:
                    continue
                row[key] = extract_field_values(val, use_normalized=use_normalized)

        # Split confidence metrics into separate columns
        conf = record.get("_confidence", {})
        if isinstance(conf, dict):
            row["biological_confidence_overall"] = conf.get("overall")
            row["biological_confidence_evidence_score"] = conf.get("evidence_score")
            row["biological_confidence_completeness"] = conf.get("completeness")
            row["biological_confidence_format_score"] = conf.get("format_score")
        else:
            row["biological_confidence_overall"] = None
            row["biological_confidence_evidence_score"] = None
            row["biological_confidence_completeness"] = None
            row["biological_confidence_format_score"] = None

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)
    print(f"Shape {result.shape} for agent path: {agent_path}")
    return result

def handle_experimental_design_agent(agent_path, use_normalized=False):
    print(f"Handling ExperimentalDesignAgent at {agent_path}")
    shard_name = os.path.basename(os.path.dirname(agent_path.rstrip(r"\/")))
    if shard_name == "NormalizedAgent":
        shard_name = os.path.basename(os.path.dirname(os.path.dirname(agent_path.rstrip(r"\/"))))
    rows = []

    for filename in sorted(os.listdir(agent_path)):
        if not filename.endswith(".json"):
            continue

        json_path = os.path.join(agent_path, filename)
        with open(json_path, "r", encoding="utf-8") as file:
            record = json.load(file)

        row = {
            "source_file": filename.split('_')[0],
            "source_shard": shard_name
        }

        # Extract values without evidence for all fields in record
        for key, val in record.items():
            if key in ["_confidence", "_hallucination_flags", "_evidence_grounding_flags"]:
                continue
            row[key] = extract_field_values(val, use_normalized=use_normalized)

        # Split confidence metrics into separate columns
        conf = record.get("_confidence", {})
        if isinstance(conf, dict):
            row["experimental_design_confidence_overall"] = conf.get("overall")
            row["experimental_design_confidence_evidence_score"] = conf.get("evidence_score")
            row["experimental_design_confidence_completeness"] = conf.get("completeness")
            row["experimental_design_confidence_format_score"] = conf.get("format_score")
        else:
            row["experimental_design_confidence_overall"] = None
            row["experimental_design_confidence_evidence_score"] = None
            row["experimental_design_confidence_completeness"] = None
            row["experimental_design_confidence_format_score"] = None

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)
    print(f"Shape {result.shape} for agent path: {agent_path}")
    return result

def handle_technical_agent(agent_path, use_normalized=False):
    print(f"Handling TechnicalAgent at {agent_path}")
    if not os.path.exists(agent_path):
        return pd.DataFrame()

    shard_name = os.path.basename(os.path.dirname(agent_path.rstrip(r"\/")))
    if shard_name == "NormalizedAgent":
        shard_name = os.path.basename(os.path.dirname(os.path.dirname(agent_path.rstrip(r"\/"))))
    rows = []

    for filename in sorted(os.listdir(agent_path)):
        if not filename.endswith(".json"):
            continue

        json_path = os.path.join(agent_path, filename)
        with open(json_path, "r", encoding="utf-8") as file:
            record = json.load(file)

        row = {
            "source_file": filename.split('_')[0],
            "source_shard": shard_name
        }

        for key, val in record.items():
            if key in ["_confidence", "_hallucination_flags", "_evidence_grounding_flags"]:
                continue
            row[key] = extract_field_values(val, use_normalized=use_normalized)

        conf = record.get("_confidence", {})
        if isinstance(conf, dict):
            row["technical_confidence_overall"] = conf.get("overall")
            row["technical_confidence_evidence_score"] = conf.get("evidence_score")
            row["technical_confidence_completeness"] = conf.get("completeness")
            row["technical_confidence_format_score"] = conf.get("format_score")
        else:
            row["technical_confidence_overall"] = None
            row["technical_confidence_evidence_score"] = None
            row["technical_confidence_completeness"] = None
            row["technical_confidence_format_score"] = None

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)
    print(f"Shape {result.shape} for agent path: {agent_path}")
    return result

def handle_normalized_agent(agent_path, filter_columns=True):
    print(f"Handling NormalizedAgent at {agent_path}")
    if not os.path.exists(agent_path):
        return pd.DataFrame()

    sub_routes = {
        "BiologicalAgent": lambda p: handle_biological_agent(p, filter_columns=filter_columns, use_normalized=True),
        "ExperimentalDesignAgent": lambda p: handle_experimental_design_agent(p, use_normalized=True),
        "TechnicalAgent": lambda p: handle_technical_agent(p, use_normalized=True),
    }

    df_norm_shard = None
    for sub in sorted(os.listdir(agent_path)):
        sub_path = os.path.join(agent_path, sub)
        if os.path.isdir(sub_path) and sub in sub_routes:
            df_sub = sub_routes[sub](sub_path)
            if df_sub is not None and not df_sub.empty:
                if df_norm_shard is None:
                    df_norm_shard = df_sub
                else:
                    merge_keys = [c for c in ["source_file", "source_shard"] if c in df_norm_shard.columns and c in df_sub.columns]
                    df_norm_shard = pd.merge(df_norm_shard, df_sub, on=merge_keys, how="outer")

    return df_norm_shard if df_norm_shard is not None else pd.DataFrame()

In [15]:
ROUTES = {"BiologicalAgent": handle_biological_agent,
          "ExperimentalDesignAgent": handle_experimental_design_agent,
          "NormalizedAgent": handle_normalized_agent,
          "TechnicalAgent": handle_technical_agent
}

def routage(agent_path):
    name = os.path.basename(agent_path)
    # action is the function corresponding to the agent type
    action = ROUTES.get(name)
    if action:
        df = action(agent_path)
    return df

In [16]:
shard_dfs = []

# Goal: Merge all raw information across agents (columns) and shards (rows)
for shard in sorted(os.listdir(PATH_HAMLET)):
    if not shard.startswith("shard_"):
        continue

    shard_path = os.path.join(PATH_HAMLET, shard)
    df_shard = None

    for agent in sorted(os.listdir(shard_path)):
        if agent == "NormalizedAgent":
            continue  # Processed separately in df_merged_normalized
        agent_path = os.path.join(shard_path, agent)
        if os.path.isdir(agent_path):
            print(f"Processing agent: {agent} in shard: {shard}")
            df_agent = routage(agent_path)
            if df_agent is not None and not df_agent.empty:
                if df_shard is None:
                    df_shard = df_agent
                else:
                    merge_keys = [c for c in ["source_file", "source_shard"] if c in df_shard.columns and c in df_agent.columns]
                    df_shard = pd.merge(df_shard, df_agent, on=merge_keys, how="outer")

    if df_shard is not None:
        shard_dfs.append(df_shard)

df_merged = pd.concat(shard_dfs, ignore_index=True) if shard_dfs else pd.DataFrame()
df_merged

Processing agent: BiologicalAgent in shard: shard_0
Handling BiologicalAgent at C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_validated\shard_0\BiologicalAgent
Shape (218, 16) for agent path: C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_validated\shard_0\BiologicalAgent
Processing agent: ExperimentalDesignAgent in shard: shard_0
Handling ExperimentalDesignAgent at C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_validated\shard_0\ExperimentalDesignAgent
Shape (218, 15) for agent path: C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_validated\shard_0\ExperimentalDesignAgent
Processing agent: TechnicalAgent in shard: shard_0
Handling TechnicalAgent at C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\q

,source_file,source_shard,age,cell_line,cell_type,developmental_stage,disease_state,material_type,sample_source,sex,...,ionization_type,labeling,mass_analyzer,ptm,reduction_concentration,reduction_reagent,technical_confidence_overall,technical_confidence_evidence_score,technical_confidence_completeness,technical_confidence_format_score
0,PXD000004,shard_0,unknown,unknown,unknown,unknown,unknown,tissue,"[brain tissue, postmortem brain tissue]",unknown,...,unknown,stable isotope-labeled neuronal proteome standard,Orbitrap,"[monohydroxylated residue, 6x(13)C labeled res...",unknown,unknown,0.886,0.972,0.500,1.0
1,PXD000445,shard_0,unknown,"[HCEC, SW480, SW620, Caco2, HT29, CX1]",colon epithelial cell,unknown,"[colorectal adenoma, normal, colorectal cancer]","[tissue, cell line]","[colonoscopy tissues, cell culture]",unknown,...,ESI,iTRAQ,Orbitrap,"[monohydroxylated residue, iTRAQ8plex-116 repo...",unknown,unknown,0.898,0.929,0.667,1.0
2,PXD001000,shard_0,unknown,unknown,unknown,unknown,Breast cancer,unknown,unknown,unknown,...,unknown,iTRAQ,Orbitrap,"[Phospho, Deamidated, iTRAQ4plex, Carbamidomet...",unknown,unknown,0.903,0.995,0.526,1.0
3,PXD001326,shard_0,unknown,ZR-75-1,"[primary human mammary fibroblasts, cancer-ass...",unknown,"[breast adenocarcinoma, breast cancer]","[tissue, cell line, primary human cells]","[tumor biopsy sections, cell culture]",unknown,...,unknown,Label-free,Orbitrap,"[Carbamidomethyl, Oxidation, Acetyl]",unknown,unknown,0.868,0.971,0.412,1.0
4,PXD001419,shard_0,unknown,unknown,"[Peripheral blood mononuclear cell, peripheral...",unknown,normal,primary cells,cell culture,unknown,...,unknown,label-free,Orbitrap,"[Carbamidomethyl, Oxidation, Acetyl]",unknown,unknown,0.861,0.944,0.444,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1732,PXD068578,shard_7,unknown,unknown,unknown,unknown,normal,cell culture,cell culture,unknown,...,unknown,label-free,Orbitrap,iodoacetamide derivatized residue,unknown,unknown,0.869,0.963,0.438,1.0
1733,PXD068828,shard_7,12 weeks,E. coli BL21 (DE3),unknown,unknown,normal,tissue,mouse liver,female,...,nanoelectrospray ionization,"[label-free quantification, label-free]","[Orbitrap, Ion Trap]","[monohydroxylated residue, acetylated residue,...","[10 mM DTT, 10 mM fresh dithiothreitol (DTT)]","[dithiothreitol (DTT), DTT]",0.950,0.930,0.926,1.0
1734,PXD069257,shard_7,8 weeks old,unknown,"[brain, blood plasma]",adult,overtraining syndrome,"[tissue, biofluid]","[mouse blood, mouse tissue]",male,...,unknown,label-free,Orbitrap,"[monohydroxylated residue, acetylated residue,...",5 mM,TCEP,0.876,0.839,0.783,1.0
1735,PXD070195,shard_7,unknown,"[Gesicle-Producer 293T cells, HEK 293T/17 cell...","[epithelial cell, fibroblast]",unknown,normal,"[cell line, tissue]","[cell culture, mouse RPE tissue]",unknown,...,ESI,label-free,Orbitrap,"[No PTMs are included in the dataset, Cysteine...",10 mM,DTT,0.961,0.967,0.889,1.0


## Normalized Agent Merge (`df_merged_normalized`)
Merge all normalized annotations across agents (`BiologicalAgent`, `ExperimentalDesignAgent`, `TechnicalAgent`) and shards.

In [17]:
norm_shards = []

# Goal: Merge all normalized information across shards
for shard in sorted(os.listdir(PATH_HAMLET)):
    if not shard.startswith("shard_"):
        continue

    norm_path = os.path.join(PATH_NORM, shard, "NormalizedAgent")
    if os.path.isdir(norm_path):
        print(f"Processing NormalizedAgent in shard: {shard}")
        df_ns = handle_normalized_agent(norm_path, filter_columns=True)
        if df_ns is not None and not df_ns.empty:
            norm_shards.append(df_ns)

df_merged_normalized = pd.concat(norm_shards, ignore_index=True) if norm_shards else pd.DataFrame()
df_merged_normalized

Processing NormalizedAgent in shard: shard_0
Handling NormalizedAgent at C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_normalization_v2_20260828\shard_0\NormalizedAgent
Handling BiologicalAgent at C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_normalization_v2_20260828\shard_0\NormalizedAgent\BiologicalAgent
Shape (218, 16) for agent path: C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_normalization_v2_20260828\shard_0\NormalizedAgent\BiologicalAgent
Handling ExperimentalDesignAgent at C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_abstract_methods_pride_multivalue_1737_normalization_v2_20260828\shard_0\NormalizedAgent\ExperimentalDesignAgent
Shape (218, 15) for agent path: C:\Users\jung.arnaud\Git_Folder\textmining\mlmarker_hamlet\input\qwen3_8_27b_ab

,source_file,source_shard,age,cell_line,cell_type,developmental_stage,disease_state,material_type,sample_source,sex,...,ionization_type,labeling,mass_analyzer,ptm,reduction_concentration,reduction_reagent,technical_confidence_overall,technical_confidence_evidence_score,technical_confidence_completeness,technical_confidence_format_score
0,PXD000004,shard_0,"{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'tissue', 'ontology_name': False}","[{'value': 'brain tissue', 'ontology_name': 'b...","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}",{'value': 'stable isotope-labeled neuronal pro...,"{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'monohydroxylated residue', 'ontolo...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.886,0.972,0.500,1.0
1,PXD000445,shard_0,"{'value': 'unknown', 'ontology_name': False}","[{'value': 'HCEC', 'ontology_name': False}, {'...","{'value': 'colon epithelial cell', 'ontology_n...","{'value': 'unknown', 'ontology_name': False}","[{'value': 'colorectal adenoma', 'ontology_nam...","[{'value': 'tissue', 'ontology_name': False}, ...","[{'value': 'colonoscopy tissues', 'ontology_na...","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'ESI', 'ontology_name': False}","{'value': 'iTRAQ', 'ontology_name': False}","{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'monohydroxylated residue', 'ontolo...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.898,0.929,0.667,1.0
2,PXD001000,shard_0,"{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'Breast cancer', 'ontology_name': 'b...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}","{'value': 'iTRAQ', 'ontology_name': False}","{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'Phospho', 'ontology_name': 'Phosph...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.903,0.995,0.526,1.0
3,PXD001326,shard_0,"{'value': 'unknown', 'ontology_name': False}","{'value': 'ZR-75-1', 'ontology_name': 'ZR-75-1...",[{'value': 'primary human mammary fibroblasts'...,"{'value': 'unknown', 'ontology_name': False}","[{'value': 'breast adenocarcinoma', 'ontology_...","[{'value': 'tissue', 'ontology_name': False}, ...","[{'value': 'tumor biopsy sections', 'ontology_...","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}","{'value': 'Label-free', 'ontology_name': False}","{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'Carbamidomethyl', 'ontology_name':...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.868,0.971,0.412,1.0
4,PXD001419,shard_0,"{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",[{'value': 'Peripheral blood mononuclear cell'...,"{'value': 'unknown', 'ontology_name': False}","{'value': 'normal', 'ontology_name': 'normal'}","{'value': 'primary cells', 'ontology_name': Fa...","{'value': 'cell culture', 'ontology_name': 'ce...","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}","{'value': 'label-free', 'ontology_name': False}","{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'Carbamidomethyl', 'ontology_name':...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.861,0.944,0.444,1.0
...,...,...,...,...,...,...,...,...,...

## Comparison: Non-Normalized vs Normalized Data
Comparison statistics showing the number and percentage of terms normalized per column, along with concrete examples.

In [18]:
# Comparison between non-normalized (df_merged) and normalized (df_merged_normalized)
common_cols = [
    c for c in df_merged.columns 
    if c in df_merged_normalized.columns and not c.startswith("source_") and "confidence" not in c
]

def has_normalization(cell_val):
    if isinstance(cell_val, list):
        return any(isinstance(x, dict) and x.get("ontology_name") is not False and x.get("ontology_name") is not None for x in cell_val)
    elif isinstance(cell_val, dict):
        return cell_val.get("ontology_name") is not False and cell_val.get("ontology_name") is not None
    return False

comparison_stats = []
diff_samples = []

for col in common_cols:
    s_raw = df_merged[col]
    s_norm = df_merged_normalized[col]
    
    is_normalized = s_norm.apply(has_normalization)
    n_norm = int(is_normalized.sum())
    pct_norm = round((n_norm / len(df_merged)) * 100, 2)
    
    comparison_stats.append({
        "Column": col,
        "Total Projects": len(df_merged),
        "Normalized Count": n_norm,
        "Normalization Rate (%)": pct_norm
    })
    
    # Extract sample differences
    diff_pxds = df_merged.loc[is_normalized, ["source_file", col]].head(3)
    for _, r in diff_pxds.iterrows():
        raw_v = r[col]
        norm_v = df_merged_normalized.loc[df_merged_normalized["source_file"] == r["source_file"], col].values[0]
        diff_samples.append({
            "PXD": r["source_file"],
            "Column": col,
            "Raw Value": raw_v,
            "Normalized (Value + Ontology Name)": norm_v
        })

df_comp_summary = pd.DataFrame(comparison_stats).sort_values(by="Normalized Count", ascending=False)
df_diff_examples = pd.DataFrame(diff_samples)

print("=== Summary of Normalization Changes per Column ===")
display(df_comp_summary)

print("\n=== Sample of Normalized Differences ===")
display(df_diff_examples.head(15))

=== Summary of Normalization Changes per Column ===


,Column,Total Projects,Normalized Count,Normalization Rate (%)
8,species,1737,1736,99.94
27,instrument,1737,1735,99.88
6,sample_source,1737,1414,81.40
31,ptm,1737,1401,80.66
4,disease_state,1737,1381,79.50
2,cell_type,1737,1226,70.58
9,tissue,1737,1101,63.39
1,cell_line,1737,523,30.11
0,age,1737,0,0.00
25,fractionation_method,1737,0,0.00



=== Sample of Normalized Differences ===


,PXD,Column,Raw Value,Normalized (Value + Ontology Name)
0,PXD000445,cell_line,"[HCEC, SW480, SW620, Caco2, HT29, CX1]","[{'value': 'HCEC', 'ontology_name': False}, {'..."
1,PXD001326,cell_line,ZR-75-1,"{'value': 'ZR-75-1', 'ontology_name': 'ZR-75-1..."
2,PXD002687,cell_line,HeLa,"{'value': 'HeLa', 'ontology_name': 'HeLa cell'}"
3,PXD000445,cell_type,colon epithelial cell,"{'value': 'colon epithelial cell', 'ontology_n..."
4,PXD001326,cell_type,"[primary human mammary fibroblasts, cancer-ass...",[{'value': 'primary human mammary fibroblasts'...
5,PXD001419,cell_type,"[Peripheral blood mononuclear cell, peripheral...",[{'value': 'Peripheral blood mononuclear cell'...
6,PXD000445,disease_state,"[colorectal adenoma, normal, colorectal cancer]","[{'value': 'colorectal adenoma', 'ontology_nam..."
7,PXD001000,disease_state,Breast cancer,"{'value': 'Breast cancer', 'ontology_name': 'b..."
8,PXD001326,disease_state,"[breast adenocarcinoma, breast cancer]","[{'value': 'breast adenocarcinoma', 'ontology_..."
9,PXD000004,sample_source,"[brain tissue, postmortem brain tissue]","[{'value': 'brain tissue', 'ontology_name': 'b..."


In [19]:
def filter_brain_insect(cell):
    if isinstance(cell, dict):
        return str(cell.get('value', '')).lower() == 'brain' and cell.get('ontology_name') == 'insect adult cerebral ganglion'
    elif isinstance(cell, list):
        return any(
            isinstance(x, dict) and str(x.get('value', '')).lower() == 'brain' and x.get('ontology_name') == 'insect adult cerebral ganglion'
            for x in cell
        )
    return False

# Filtrer toutes les lignes correspondantes
df_brain_insect = df_merged_normalized[df_merged_normalized['tissue'].apply(filter_brain_insect)]
print(f"Number of lines found: {len(df_brain_insect)}")
df_brain_insect

Number of lines found: 0


,source_file,source_shard,age,cell_line,cell_type,developmental_stage,disease_state,material_type,sample_source,sex,...,ionization_type,labeling,mass_analyzer,ptm,reduction_concentration,reduction_reagent,technical_confidence_overall,technical_confidence_evidence_score,technical_confidence_completeness,technical_confidence_format_score


In [20]:
# Filtration in one single line with Pandas functions 
df_tissue_brain = df_merged_normalized[
    df_merged_normalized['tissue'].astype(str).str.contains(r"'value':\s*'brain'", case=False, na=False)
]
df_tissue_brain


,source_file,source_shard,age,cell_line,cell_type,developmental_stage,disease_state,material_type,sample_source,sex,...,ionization_type,labeling,mass_analyzer,ptm,reduction_concentration,reduction_reagent,technical_confidence_overall,technical_confidence_evidence_score,technical_confidence_completeness,technical_confidence_format_score
0,PXD000004,shard_0,"{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'tissue', 'ontology_name': False}","[{'value': 'brain tissue', 'ontology_name': 'b...","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}",{'value': 'stable isotope-labeled neuronal pro...,"{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'monohydroxylated residue', 'ontolo...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.886,0.972,0.500,1.0
20,PXD005753,shard_0,"{'value': '6 months', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'neurons', 'ontology_name': 'neuron'}","{'value': 'adult', 'ontology_name': False}","{'value': 'Huntington disease', 'ontology_name...","{'value': 'tissue', 'ontology_name': False}","[{'value': 'mouse brain', 'ontology_name': 'br...","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'ESI', 'ontology_name': False}","[{'value': 'TMT', 'ontology_name': False}, {'v...","{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'methionine oxidation', 'ontology_n...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.900,0.892,0.769,1.0
25,PXD006607,shard_0,"{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'primary human medulloblastomas', 'o...","{'value': 'unknown', 'ontology_name': False}","{'value': 'Childhood medulloblastoma', 'ontolo...","{'value': 'tissue', 'ontology_name': False}","{'value': 'primary human medulloblastomas', 'o...","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}","{'value': 'SILAC', 'ontology_name': False}","{'value': 'Orbitrap', 'ontology_name': False}","{'value': 'phosphorylated residue', 'ontology_...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.863,0.993,0.333,1.0
49,PXD009578,shard_0,"[{'value': '2 years, range 1–8.5 years', 'onto...","{'value': 'unknown', 'ontology_name': False}","{'value': 'Dendritic cell', 'ontology_name': '...","{'value': 'postnatal', 'ontology_name': False}","{'value': 'Neuroblastoma', 'ontology_name': 'n...","[{'value': 'tissue', 'ontology_name': False}, ...","{'value': 'rat cerebellum', 'ontology_name': '...","{'value': 'male', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}","[{'value': 'Protein Abundance Index - PAI', 'o...","{'value': 'Orbitrap', 'ontology_name': False}",{'value': 'No PTMs are included in the dataset...,"{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.829,0.894,0.412,1.0
50,PXD009646,shard_0,"{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}","[{'value': 'normal', 'ontology_name': 'normal'...","{'value': 'biofluid', 'ontology_name': False}","{'value': 'donor CSF', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",...,"{'value': 'unknown', 'ontology_name': False}","{'value': 'TMT', 'ontology_name': False}","{'value': 'Orbitrap', 'ontology_name': False}","[{'value': 'Ammonia-loss', 'ontology_name': 'A...","{'value': 'unknown', 'ontology_name': False}","{'value': 'unknown', 'ontology_name': False}",0.912,0.995,0.571,1.0
...,...,...,...,...,...,...,...,...,...,...,..

## 3-Way Comparisons Tissus : HAMLET (`df_merged`, `df_merged_normalized`) vs LLM Approach (Sanders' one) (`df_llm`)
Annotation tissus comparison between both pipeline:
1. **`df_merged` (HAMLET Raw)**: Multi-agent HAMLET pipeline without standardized ontologies.
2. **`df_merged_normalized` (HAMLET Normalisé)**: Multi-agent HAMLET pipeline with standardized ontologies. (UBERON / BTO / CL).
3. **`df_llm` (Approche LLM)**: Multi-agent LLM pipeline for tissue predictions. (`agent_metadata.tsv`).

In [21]:
# 1. Loading LLM approach (df_llm)
PATH_LLM = r"C:\Users\jung.arnaud\Project\HAMLET_MLMarker\Input\agentic-metadata\results\agent_metadata.tsv"
df_llm = pd.read_csv(PATH_LLM, sep="\t").drop_duplicates(subset=["pxd"]).copy()
df_llm_map = df_llm.set_index("pxd")["tissue"].to_dict()

# 2. Extraction and Cleaning Functions
def extract_raw_tissue(val):
    if val is None:
        return None
    if isinstance(val, (list, tuple, set)):
        items = [str(x).strip() for x in val if x and str(x).lower() != "unknown"]
        return items if len(items) > 1 else (items[0] if items else None)
    if pd.isna(val) or str(val).strip().lower() == "unknown":
        return None
    return str(val).strip()

def extract_norm_tissue(cell):
    if cell is None:
        return None
    if isinstance(cell, dict):
        ont = cell.get("ontology_name")
        return ont if ont is not False and ont is not None else cell.get("value")
    elif isinstance(cell, list):
        items = []
        for x in cell:
            if isinstance(x, dict):
                ont = x.get("ontology_name")
                items.append(ont if ont is not False and ont is not None else x.get("value"))
            else:
                items.append(str(x))
        valid = [i for i in items if i and str(i).lower() != "unknown"]
        return valid if len(valid) > 1 else (valid[0] if valid else None)
    return str(cell)

def format_clean_str(val):
    if val is None:
        return ""
    if isinstance(val, (list, tuple, set)):
        items = sorted([str(x).strip().lower() for x in val if str(x).strip().lower() not in ["unknown", "none", "nan", ""]])
        return "; ".join(items)
    if pd.isna(val) or str(val).strip().lower() in ["unknown", "none", "nan", ""]:
        return ""
    return str(val).strip().lower()

# 3. Building the Comparative DataFrame
df_3way = pd.DataFrame()
df_3way["pxd"] = df_merged["source_file"]
df_3way["source_shard"] = df_merged["source_shard"]
df_3way["tissue_hamlet_raw"] = df_merged["tissue"].apply(extract_raw_tissue)
df_3way["tissue_hamlet_norm"] = df_merged_normalized["tissue"].apply(extract_norm_tissue)
df_3way["tissue_llm"] = df_3way["pxd"].map(df_llm_map)

# String representation standardized
df_3way["str_hamlet_raw"] = df_3way["tissue_hamlet_raw"].apply(format_clean_str)
df_3way["str_hamlet_norm"] = df_3way["tissue_hamlet_norm"].apply(format_clean_str)
df_3way["str_llm"] = df_3way["tissue_llm"].apply(format_clean_str)

df_3way["has_hamlet_raw"] = df_3way["str_hamlet_raw"] != ""
df_3way["has_hamlet_norm"] = df_3way["str_hamlet_norm"] != ""
df_3way["has_llm"] = df_3way["str_llm"] != ""

# 5. Recap Table of Annotation Coverage
coverage_summary = pd.DataFrame([
    {"Pipeline": "HAMLET (df_merged Brut)", "Annotated projects": int(df_3way["has_hamlet_raw"].sum()), "Coverage (%)": round(df_3way["has_hamlet_raw"].mean() * 100, 2)},
    {"Pipeline": "HAMLET (df_merged_normalized)", "Annotated projects": int(df_3way["has_hamlet_norm"].sum()), "Coverage (%)": round(df_3way["has_hamlet_norm"].mean() * 100, 2)},
    {"Pipeline": "Approche LLM (df_llm)", "Annotated projects": int(df_3way["has_llm"].sum()), "Coverage (%)": round(df_3way["has_llm"].mean() * 100, 2)},
    {"Pipeline": "Co-annotés par HAMLET et LLM", "Annotated projects": int((df_3way["has_hamlet_raw"] & df_3way["has_llm"]).sum()), "Coverage (%)": round((df_3way["has_hamlet_raw"] & df_3way["has_llm"]).mean() * 100, 2)}
])

print("=== 1. Comparison of Annotation Coverage (Total N = {}) ===".format(len(df_3way)))
display(coverage_summary)

# 5. Metrics of Concordance on the Common Projects
co_annot = df_3way[df_3way["has_hamlet_raw"] & df_3way["has_llm"]].copy()

def partial_overlap(s1, s2):
    if not s1 or not s2:
        return False
    if s1 == s2:
        return True
    p1 = [p.strip() for p in s1.split(";") if p.strip()]
    p2 = [p.strip() for p in s2.split(";") if p.strip()]
    return any(a in b or b in a for a in p1 for b in p2)

co_annot["all_3_equal"] = (co_annot["str_hamlet_raw"] == co_annot["str_hamlet_norm"]) & (co_annot["str_hamlet_norm"] == co_annot["str_llm"])
co_annot["hamlet_raw_vs_llm_equal"] = co_annot["str_hamlet_raw"] == co_annot["str_llm"]
co_annot["hamlet_norm_vs_llm_equal"] = co_annot["str_hamlet_norm"] == co_annot["str_llm"]
co_annot["hamlet_vs_llm_partial"] = co_annot.apply(lambda r: partial_overlap(r["str_hamlet_raw"], r["str_llm"]), axis=1)

concordance_summary = pd.DataFrame([
    {"Metric": "Perfect Consensus (Raw HAMLET == Normalized HAMLET == LLM)", "Projects": int(co_annot["all_3_equal"].sum()), "Rate (%)": round(co_annot["all_3_equal"].mean() * 100, 2)},
    {"Metric": "Exact Agreement: Raw HAMLET (df_merged) == LLM (df_llm)", "Projects": int(co_annot["hamlet_raw_vs_llm_equal"].sum()), "Rate (%)": round(co_annot["hamlet_raw_vs_llm_equal"].mean() * 100, 2)},
    {"Metric": "Exact Agreement: Normalized HAMLET == LLM (df_llm)", "Projects": int(co_annot["hamlet_norm_vs_llm_equal"].sum()), "Rate (%)": round(co_annot["hamlet_norm_vs_llm_equal"].mean() * 100, 2)},
    {"Metric": "Partial Agreement / Anatomical Inclusion (HAMLET vs LLM)", "Projects": int(co_annot["hamlet_vs_llm_partial"].sum()), "Rate (%)": round(co_annot["hamlet_vs_llm_partial"].mean() * 100, 2)},
    {"Metric": "Changes due to Normalization in HAMLET (Raw != Norm)", "Projects": int((~df_3way["str_hamlet_raw"].eq(df_3way["str_hamlet_norm"]) & df_3way["has_hamlet_raw"]).sum()), "Rate (%)": round((~df_3way["str_hamlet_raw"].eq(df_3way["str_hamlet_norm"]) & df_3way["has_hamlet_raw"]).mean() * 100, 2)}
])

print("\n=== 2. Concordance and Similarities across Shared Projects (N = {}) ===".format(len(co_annot)))
display(concordance_summary)

# 5. Side-by-Side Comparative Sample
print("\n=== 3. Side-by-Side Comparative Sample ===")
display(co_annot[["pxd", "tissue_hamlet_raw", "tissue_hamlet_norm", "tissue_llm"]].head(15))

=== 1. Comparison of Annotation Coverage (Total N = 1737) ===


,Pipeline,Annotated projects,Coverage (%)
0,HAMLET (df_merged Brut),1241,71.45
1,HAMLET (df_merged_normalized),1241,71.45
2,Approche LLM (df_llm),705,40.59
3,Co-annotés par HAMLET et LLM,609,35.06



=== 2. Concordance and Similarities across Shared Projects (N = 609) ===


,Metric,Projects,Rate (%)
0,Perfect Consensus (Raw HAMLET == Normalized HA...,248,40.72
1,Exact Agreement: Raw HAMLET (df_merged) == LLM...,272,44.66
2,Exact Agreement: Normalized HAMLET == LLM (df_...,248,40.72
3,Partial Agreement / Anatomical Inclusion (HAML...,464,76.19
4,Changes due to Normalization in HAMLET (Raw !=...,137,7.89



=== 3. Side-by-Side Comparative Sample ===


,pxd,tissue_hamlet_raw,tissue_hamlet_norm,tissue_llm
0,PXD000004,brain,brain,brain
1,PXD000445,colon,colon,colon
3,PXD001326,Breast,breast,breast
4,PXD001419,Blood,blood,PBMCs
5,PXD001524,Testis,testis,testis
6,PXD001985,"[lung, Sputum]","[lung, sputum]",sputum
12,PXD003416,skin,zone of skin,skin
14,PXD003971,skin,zone of skin,skin
19,PXD005544,breast,breast,breast
20,PXD005753,brain,brain,brain


In [22]:
os.makedirs(PATH_OUTPUT, exist_ok=True)
df_merged_normalized.to_csv(os.path.join(PATH_OUTPUT, 'HAMLET_normalized.tsv'), sep='\t', index=False)
df_merged.to_csv(os.path.join(PATH_OUTPUT, 'HAMLET.tsv'), sep='\t', index=False)